# Testing LLM APIs
Welcome to this practical session. In this notebook, we will explore how to interact with the production OpenAI API, manipulate hyperparameters like temperature, and build automated testing assertions for validating structured responses.

In [ ]:
import os
import os.path as osp

from pathlib import Path
from pprint import pprint

import sys

root = Path.cwd().parent 
if str(root) not in sys.path:    
    sys.path.append(str(root))

In [ ]:
from src.config import CFG

## Installation & Environment Setup
First, we need to install the official OpenAI SDK and configure our secure API token environment variable.

In [ ]:
import time
import os
import json
from openai import OpenAI
import os 

# Instruct students to input their real OpenAI API key

# Initialize the standard production client
client = OpenAI(api_key=CFG.OPENAI_API_KEY)

print("OpenAI Production Client Initialized!")

In [ ]:
from pprint import pprint

## Managing Hyperparameters: Temperature Determinism
Let's see how setting a low temperature (highly deterministic) vs. a high temperature (highly creative) impacts raw API output variability.

In [ ]:
prompt = "Qu'est-ce qu'un agent IA? réponds en quelques lignes"

outputs = []
print("--- Testing Deterministic Output (temperature=0.0) ---")
for i in range(2):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.
    )
    print(f"Run {i+1}:\n")
    response = response.choices[0].message.content.strip()
    pprint(response)
    outputs.append(response)
    print()

In [ ]:
from src.utils import calc_similarity

In [ ]:
calc_similarity(outputs[0], outputs[1])

In [ ]:
print("\n--- Testing Creative Output (temperature=1.2) ---")
outputs = []

for i in range(3):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=1.2
    )
    print(f"Run {i+1}:\n")
    response = response.choices[0].message.content.strip()
    pprint(response)
    outputs.append(response)
    print()

In [ ]:
calc_similarity(outputs[0], outputs[1])

In [ ]:
response = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[{"role": "user", "content": prompt}],
        # temperature=0.3,
        reasoning_effort="low"
    )

response = response.choices[0].message.content.strip()
print(response)


In [ ]:
prompt = "Qu'est-ce qu'un agent IA? réponds avec 300 mots"

response = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": prompt}],
    reasoning_effort="low",
    stream=True  # Enables streaming
)

for chunk in response:
    content = chunk.choices[0].delta.content
    if content:
        print(content, end="", flush=True)

print() 